# Lab 5: Workflows Sequenciais (10 min)

## Objetivos
- Entender o conceito de workflow com agentes
- Criar um pipeline sequencial (agente → agente)
- Ver como encadear tarefas de IA

## 📖 O que é um Workflow Sequencial?

Um **workflow sequencial** encadeia múltiplos passos de IA, onde o output de um passo é o input do seguinte.

```
Texto original
      │
      ▼
┌─────────────┐
│ Passo 1:     │
│ Resumir      │
└──────┬──────┘
       │ resumo
       ▼
┌─────────────┐
│ Passo 2:     │
│ Traduzir     │
└──────┬──────┘
       │ tradução
       ▼
┌─────────────┐
│ Passo 3:     │
│ Formatar     │
└──────┬──────┘
       │
       ▼
  Resultado final
```

### Quando usar?
- Tarefas complexas que beneficiam de ser divididas
- Cada passo precisa de instruções específicas
- Queres controlar/validar cada etapa

In [ ]:
# Setup
from dotenv import load_dotenv
import os

load_dotenv("../.env")

from openai import AzureOpenAI

client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2025-01-01-preview"),
)

DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o")
print("✅ Setup concluído!")

In [ ]:
# Helper: chamar o modelo com um passo do workflow
def executar_passo(nome_passo: str, system_prompt: str, user_input: str) -> str:
    """Executa um passo do workflow e retorna o resultado."""
    print(f"\n{'='*50}")
    print(f"🔄 Passo: {nome_passo}")
    print(f"{'='*50}")
    
    response = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_input},
        ],
        temperature=0.3,
        max_tokens=500,
    )
    
    resultado = response.choices[0].message.content
    print(f"📤 Output: {resultado[:200]}..." if len(resultado) > 200 else f"📤 Output: {resultado}")
    return resultado

## 💻 Exercício 5.1: Pipeline de Processamento de Texto

Vamos criar um workflow de 3 passos:
1. **Resumir** um texto longo
2. **Traduzir** o resumo para inglês
3. **Gerar** um tweet com base na tradução

In [ ]:
# Texto original (longo)
texto_original = """
A inteligência artificial tem vindo a transformar diversos setores da economia portuguesa. 
No setor da saúde, hospitais em Lisboa e Porto já utilizam sistemas de IA para auxiliar no 
diagnóstico de doenças, análise de exames médicos e planeamento de tratamentos. No setor 
financeiro, os bancos portugueses implementaram chatbots inteligentes para atendimento ao 
cliente e sistemas de deteção de fraude baseados em machine learning. A agricultura também 
não ficou para trás, com startups portuguesas a desenvolver drones equipados com IA para 
monitorização de culturas e otimização de irrigação. O governo português lançou recentemente 
a Estratégia Nacional de IA, com um investimento de 200 milhões de euros até 2026, focada 
na formação de talento, infraestrutura tecnológica e regulação ética. Especialistas preveem 
que a IA poderá contribuir com mais de 30 mil milhões de euros para o PIB português até 2030, 
criando milhares de novos empregos qualificados e transformando fundamentalmente a forma 
como as empresas portuguesas operam e competem no mercado global.
"""

print(f"📄 Texto original ({len(texto_original.split())} palavras):")
print(texto_original)

In [ ]:
# PASSO 1: Resumir
resumo = executar_passo(
    nome_passo="Resumir",
    system_prompt="Resume o texto fornecido em 2-3 frases concisas em português. Mantém os dados numéricos mais importantes.",
    user_input=texto_original,
)

# PASSO 2: Traduzir para inglês
traducao = executar_passo(
    nome_passo="Traduzir para Inglês",
    system_prompt="Traduz o texto seguinte para inglês. Mantém um tom profissional.",
    user_input=resumo,
)

# PASSO 3: Gerar tweet
tweet = executar_passo(
    nome_passo="Gerar Tweet",
    system_prompt="Com base no texto, cria um tweet (máximo 280 caracteres) em inglês. Deve ser cativante e incluir 2-3 hashtags relevantes.",
    user_input=traducao,
)

print(f"\n{'='*50}")
print("🏁 RESULTADO FINAL")
print(f"{'='*50}")
print(f"🐦 Tweet: {tweet}")
print(f"📏 Caracteres: {len(tweet)}")

## 💻 Exercício 5.2: Pipeline de Análise de Feedback

Workflow de 3 passos para processar feedback de clientes:
1. **Classificar** o sentimento
2. **Extrair** os temas principais
3. **Gerar** resposta sugerida

In [ ]:
import json

feedback_cliente = """
Estou muito desiludido com o serviço de apoio ao cliente. Liguei 3 vezes esta semana e 
ninguém resolveu o meu problema com a fatura. O tempo de espera foi superior a 30 minutos 
em cada chamada. No entanto, o último operador que me atendeu, o Carlos, foi muito prestável 
e parece que finalmente encaminhou a situação. Gostava que fosse mais fácil resolver problemas 
sem ter de ligar tantas vezes.
"""

# PASSO 1: Classificar sentimento
sentimento = executar_passo(
    nome_passo="Classificar Sentimento",
    system_prompt="""Analisa o sentimento do feedback. Responde APENAS em JSON com este formato:
{"sentimento": "positivo|negativo|misto", "score": 0.0-1.0, "justificacao": "breve"}""",
    user_input=feedback_cliente,
)

# PASSO 2: Extrair temas
temas = executar_passo(
    nome_passo="Extrair Temas",
    system_prompt="""Extrai os temas principais do feedback. Responde APENAS em JSON:
{"temas": ["tema1", "tema2"], "urgencia": "alta|media|baixa", "departamento": "qual o departamento responsável"}""",
    user_input=feedback_cliente,
)

# PASSO 3: Gerar resposta
resposta_sugerida = executar_passo(
    nome_passo="Gerar Resposta ao Cliente",
    system_prompt="""Com base na análise do feedback, gera uma resposta profissional e empática ao cliente.
A resposta deve reconhecer o problema, agradecer o feedback, e indicar próximos passos.""",
    user_input=f"Feedback: {feedback_cliente}\nAnálise sentimento: {sentimento}\nTemas: {temas}",
)

print(f"\n{'='*50}")
print("📧 RESPOSTA SUGERIDA AO CLIENTE:")
print(f"{'='*50}")
print(resposta_sugerida)

## ✅ Resumo

Neste lab aprendeste:
- O conceito de workflows sequenciais com IA
- Como encadear chamadas ao modelo (output → input)
- Pipelines práticos: processamento de texto e análise de feedback

**Próximo:** [Lab 6 - APIM](lab06-apim.ipynb)